# İş Yatırım Hisse Tarayıcı API

**İş Yatırım Gelişmiş Hisse Arama** — parametrik kriter aralıklarıyla BIST hisse taraması (getiri potansiyeli, kapanış, hedef fiyat, piyasa değeri, F/K, ...).

> Public endpoint, auth yok. Yanıt `d` alanı içinde **stringified JSON** olarak gelir — iki aşamalı parse gerekir.

## Endpoint

```
POST https://www.isyatirim.com.tr/tr-tr/analiz/_Layouts/15/IsYatirim.Website/StockInfo/CompanyInfoAjax.aspx/getScreenerDataNEW
```

**Zorunlu header'lar:**

| Header | Değer |
|--------|-------|
| `Content-Type` | `application/json; charset=UTF-8` |
| `X-Requested-With` | `XMLHttpRequest` |
| `Referer` | `https://www.isyatirim.com.tr/tr-tr/analiz/hisse/Sayfalar/gelismis-hisse-arama.aspx` |

Auth: yok. `Referer` veya `X-Requested-With` eksikse 500/403 dönebilir.

## İstek (Request) Alanları

| Alan | Açıklama |
|------|----------|
| `sektor` | Sektör kodu (4 hane) · `""` = tümü |
| `endeks` | Endeks kodu · `""` = tümü |
| `takip` | Takip listesi filtresi |
| `oneri` | Analist önerisi filtresi |
| `criterias` | `[[kod, min, max, "False"], ...]` ← parametrik aralık listesi |
| `lang` | `1055` = TR · `1033` = EN |

### `endeks` kodları

| Kod | Endeks |
|-----|--------|
| `""` | Tümü |
| `03` | BIST 30 |
| `05` | BIST 50 |
| `01` | BIST 100 |
| `09` | Ulusal-Tüm |

### `oneri` kodları

| Kod | Anlam |
|-----|-------|
| `""` | Tümü |
| `AL` | AL |
| `SAT` | SAT |
| `TUT` | TUT |
| `GÖZDEN GEÇİRİLİYOR` | Gözden Geçiriliyor |
| `ÖNERİMİZ YOK` | Önerimiz Yok |

### `takip` kodları

| Kod | Anlam |
|-----|-------|
| `""` | Tümü |
| `1` | Takip listesinde |
| `0` | Takip listesinde değil |

### `lang` kodları

| Kod | Dil |
|-----|-----|
| `1055` | Türkçe |
| `1033` | İngilizce |

### Öne Çıkan Kriter Kodları

| Kod | Anlam |
|-----|-------|
| `61` | Getiri Potansiyeli (%) — min/max senin parametren |
| `7` | Kapanış (TL) — fiyat aralığı filtresi |
| `51` | Hedef Fiyat (TL) |

> Tüm kriter kodlarının listesi için DM. Her kriter `["<kod>", "<min>", "<max>", "False"]` formatında 4 elemanlı dizi olarak gönderilir.

## Yanıt (Response) Yapısı

Dış zarf:

```json
{ "d": "<JSON string>" }
```

`d` alanı **string** olarak JSON tutar — `json.loads(r.json()['d'])` ile iki aşamalı parse edilir.

Parse sonrası: `Array<{ Hisse, <sorulan kriter kodları>... }>` — her satır bir hissedir, anahtarlar criteria kodlarıdır.

### Her satırda dönen alanlar

| Alan | Örnek |
|------|-------|
| `Hisse` | `"THYAO - Türk Hava Yolları"` |
| `"7"` | `"329"` (Kapanış TL, string) |
| `"61"` | `"76.29"` (Getiri Potansiyeli %, string) |
| `"51"` | `"580.50"` (Hedef Fiyat TL, string) |
| `"8"` | `"455000"` (Piyasa Değeri mn TL, string) |
| `"28"` | `"12.4"` (Cari F/K, string) |

Yanıtta sadece `criterias` body'sinde sorulan kriter kodları gelir.

## 1. Ortak Ayarlar

In [1]:
import requests

URL = (
    "https://www.isyatirim.com.tr/tr-tr/analiz/_Layouts/15/"
    "IsYatirim.Website/StockInfo/CompanyInfoAjax.aspx/getScreenerDataNEW"
)

HEADERS = {
    "Content-Type": "application/json; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": "https://www.isyatirim.com.tr/tr-tr/analiz/hisse/Sayfalar/gelismis-hisse-arama.aspx",
}


def num(s):
    """String veya None → float (None/boş → NaN). API tüm sayıları string döner."""
    if s is None or s == "":
        return float("nan")
    return float(s)

## 2. Tarama Fonksiyonu

`criterias`, her biri `[kod, min, max, "False"]` olan dizi listesidir. Yanıttaki `d` stringini ikinci kez `json.loads` ile parse ederiz.

In [2]:
import json


def tara(sektor="", endeks="", takip="", oneri="", criterias=None, lang="1055"):
    body = {
        "sektor": sektor,
        "endeks": endeks,
        "takip": takip,
        "oneri": oneri,
        "criterias": criterias or [],
        "lang": lang,
    }
    r = requests.post(URL, headers=HEADERS, json=body)
    r.raise_for_status()
    return json.loads(r.json()["d"])

## 3. AL önerili · BIST-30 · Getiri Potansiyeli Taraması

Image'daki örnek istek: `endeks=03` (BIST 30), `oneri=AL`, kapanış 1–50.000 TL, getiri potansiyeli −100…500%.

In [3]:
rows = tara(
    endeks="03",
    oneri="AL",
    criterias=[
        ["7", "1", "50000", "False"],     # kapanış aralığı (TL)
        ["61", "-100", "500", "False"],   # getiri potansiyeli (%)
    ],
)

rows.sort(key=lambda r: num(r.get("61")), reverse=True)

print(f"AL önerili BIST-30 · {len(rows)} hisse · getiri potansiyeline göre azalan\n")
print(f"{'Hisse':<35} {'Kapanış':>10} {'Getiri Pot.':>14}")
print("-" * 62)
for row in rows:
    print(f"{row['Hisse'].strip():<35} {num(row.get('7')):>10.2f} {num(row.get('61')):>13.2f}%")

AL önerili BIST-30 · 22 hisse · getiri potansiyeline göre azalan

Hisse                                  Kapanış    Getiri Pot.
--------------------------------------------------------------
SAHOL - Sabancı Holding                  98.15         84.41%
MGROS - Migros                          634.00         67.47%
AEFES - Anadolu Efes                     19.24         64.24%
EKGYO - Emlak Konut GYO                  21.00         60.70%
KCHOL - Koç Holding                     205.00         50.24%
TOASO - Tofaş Fabrika                   297.00         48.97%
FROTO - Ford Otosan                     102.60         48.63%
TCELL - Turkcell                        116.60         43.77%
GARAN - Garanti Bankası                 136.70         42.65%
THYAO - Türk Hava Yolları               320.50         41.97%
TTKOM - Türk Telekom                     64.30         38.03%
AKBNK - Akbank                           77.70         37.71%
VAKBN - Vakıfbank                        32.92         36.69%
TAV

## 4. Hedef Fiyat Dahil · BIST-100 · TUT + AL

Aynı tarama ama `endeks=01` (BIST 100), `oneri=""` (tümü), kriter listesinde `51` (hedef fiyat) de var.

In [4]:
rows = tara(
    endeks="01",
    criterias=[
        ["7", "1", "50000", "False"],
        ["61", "0", "500", "False"],     # sadece pozitif getiri potansiyelliler
        ["51", "0", "100000", "False"],  # hedef fiyatı bilinenler
    ],
)

rows.sort(key=lambda r: num(r.get("61")), reverse=True)

print(f"BIST-100 · pozitif getiri potansiyelli · {len(rows)} hisse · ilk 15\n")
print(f"{'Hisse':<35} {'Kapanış':>10} {'Hedef':>10} {'Getiri Pot.':>14}")
print("-" * 73)
for row in rows[:15]:
    print(f"{row['Hisse'].strip():<35} {num(row.get('7')):>10.2f} {num(row.get('51')):>10.2f} {num(row.get('61')):>13.2f}%")

BIST-100 · pozitif getiri potansiyelli · 44 hisse · ilk 15

Hisse                                  Kapanış      Hedef    Getiri Pot.
-------------------------------------------------------------------------
SOKM - Şok Marketler                     49.36      98.19         98.93%
ALARK - Alarko Holding                   95.10     184.89         94.42%
SAHOL - Sabancı Holding                  98.15     181.00         84.41%
ANSGR - Anadolu Sigorta                  28.68      49.88         73.91%
ULKER - Ülker Bisküvi                   124.00     215.00         73.39%
AGHOL - Anadolu Grubu Holding            30.90      53.00         71.52%
MGROS - Migros                          634.00    1061.74         67.47%
AEFES - Anadolu Efes                     19.24      31.60         64.24%
TURSG - Türkiye Sigorta                  13.75      22.43         63.11%
TSKB - TSKB                              12.28      20.00         62.87%
EKGYO - Emlak Konut GYO                  21.00      33.75      

## 5. curl Eşdeğeri

Aynı isteği shell'den atmak için. `jq` ile dış `d` stringini parse edip ikinci `fromjson` ile içerik tablosuna iniyoruz.

In [5]:
%%bash
curl -s -X POST \
  "https://www.isyatirim.com.tr/tr-tr/analiz/_Layouts/15/IsYatirim.Website/StockInfo/CompanyInfoAjax.aspx/getScreenerDataNEW" \
  -H "Content-Type: application/json; charset=UTF-8" \
  -H "X-Requested-With: XMLHttpRequest" \
  -H "Referer: https://www.isyatirim.com.tr/tr-tr/analiz/hisse/Sayfalar/gelismis-hisse-arama.aspx" \
  -d '{"sektor":"","endeks":"03","takip":"","oneri":"AL","criterias":[["7","1","50000","False"],["61","-100","500","False"]],"lang":"1055"}' \
  | jq -r '.d | fromjson
      | map({hisse: .Hisse, fiyat: (.["7"] | tonumber? // 0), pot: (.["61"] | tonumber? // -9999)})
      | sort_by(-.pot)
      | (["HISSE","FIYAT (TL)","GETIRI POT. (%)"]),
        (["-----","----------","----------------"]),
        (.[] | [.hisse, .fiyat, .pot])
      | @tsv' \
  | column -t -s $'\t'

HISSE                                   FIYAT (TL)  GETIRI POT. (%)
-----                                   ----------  ----------------
SAHOL - Sabancı Holding                 98.15       84.41
MGROS - Migros                          634         67.47
AEFES - Anadolu Efes                    19.24       64.24
EKGYO - Emlak Konut GYO                 21          60.7
KCHOL - Koç Holding                     205         50.24
TOASO - Tofaş Fabrika                   297         48.97
FROTO - Ford Otosan                     102.6       48.63
TCELL - Turkcell                        116.6       43.77
GARAN - Garanti Bankası                 136.7       42.65
THYAO - Türk Hava Yolları               320.5       41.97
TTKOM - Türk Telekom                    64.3        38.03
AKBNK - Akbank                          77.7        37.71
VAKBN - Vakıfbank                       32.92       36.69
TAVHL - TAV Holding                     302         34.67
BIMAS - Bim Birleşik Mağazalar A.Ş      752.5       

## Notlar

- **İki aşamalı parse:** `r.json()` → dış zarf (`{"d": ...}`) → `json.loads(r.json()["d"])` → asıl liste. Tek `r.json()` ile satıra inilemez.
- **Tüm sayılar string:** `"7": "329"`, `"61": "76.29"` — float karşılaştırma/sıralama öncesi `num()` ile çevir. US ondalık (`.`), TR formatı değil.
- **`Hisse` alanı boşluklu olabilir:** Bazı kayıtlar trailing whitespace içerir (örn. `"ASELS - Aselsan                       "`) — `strip()` uygula.
- **Kriter sözdizimi sabit:** `["<kod>", "<min>", "<max>", "False"]` — 4. eleman her zaman `"False"` string'i.
- **Yanıt sütunları dinamik:** Yanıttaki anahtarlar `criterias` body'sinde sorulan kodlarla sınırlıdır; sormadığın kriter dönmez.
- **Header zorunluluğu:** `X-Requested-With: XMLHttpRequest` ve `Referer` olmadan endpoint 500/403 verir — browser-only kontrol.
- **Auth yok:** Cookie/token gerekmez; rate-limit davranışı belgelenmemiş, polling'i makul tut.